In [110]:
symbols = {"NVDA","META","BYND","CHGG","AAPL","BRK-B","JNJ","AMZN","GOOGL","TSLA","AMD","PLTR","RIVN","COIN","AI","UPST","FSLY","OPEN","HOOD","SPY","QQQ","XLF","XLE","ARKK","SPY"} # diverse data

import yfinance as yf
import functions
import pandas as pd
from PIL import Image
from IPython.display import display
import random
from datetime import datetime
import warnings
import math
import numpy as np
# loop through symbols, use predictTest, use frequencies as seconds, use 1mo,3mo,6mo,1y,2y,5y as range

warnings.simplefilter(action='ignore', category=FutureWarning)
charts = functions.Charts()

def distribute(values:list, error:float, proximity:float):
    offset = error*proximity
    nudged = [v + random.uniform(-offset, offset) for v in values]
    nudged = [max(v, 0.001) for v in nudged]
    total = sum(nudged)
    return [float(v/total) for v in nudged]

ranges = ["2022-01-01","2025-11-30"]
dates = pd.date_range(start=ranges[0], end=ranges[1])

In [111]:
biases:dict[str,list] = {}

for symbol in symbols:
    print(symbol)
    sector = yf.Sector(yf.Ticker(symbol).info.get("sectorKey", "Unknown")).ticker.info["displayName"]
    prices = yf.download(symbol, start=ranges[0], end=ranges[1], progress=False)["Close"]
    if sector not in biases:
        biases[sector] = [[0.2, 0.2, 0.2, 0.2, 0.2], 0]
    
    bestWeight = biases[sector][0]

    for i, date in enumerate(dates):
        bestError = 9999.0
        bestProx = 0.1
        trials = 0

        while trials <= 50:
            tests:list = distribute(bestWeight,bestError,bestProx)
            bias = {90:[tests[0], "ME"], 180:[tests[1], "ME"], 365:[tests[2], "D"], 730:[tests[3], "W"], 1825:[tests[4], "YS"]}
            guess = round(charts.projectTestDay(ticker=symbol, weights=str(bias), today=date),2)
            actual = round(prices.iloc[i][0],2)
            error = abs(actual-guess)
            if error < bestError:
                bestWeight = tests
                bestError = error
                bestProx = bestError*0.01
            print(trials, error, bestError, bestProx, tests, bestWeight)
            trials += 1

        prevWeight, count = biases[sector]
        cma = [(prevWeight[j]*count+bestWeight[j])/(count + 1) for j in range(len(prevWeight))]
        biases[sector] = [cma,count+1]

        print(biases)

for s, val in biases.items():
    print(f"{s}: {val}")

NVDA
0 6.739999999999998 6.739999999999998 0.06739999999999999 [3.848126727713332e-07, 0.29610147017459754, 0.3450922758620141, 0.35880548433804277, 3.848126727713332e-07] [3.848126727713332e-07, 0.29610147017459754, 0.3450922758620141, 0.35880548433804277, 3.848126727713332e-07]
1 7.559999999999999 6.739999999999998 0.06739999999999999 [0.0005396346235836145, 0.32985928807658516, 0.3017746731026409, 0.20690314607937552, 0.16092325811781483] [3.848126727713332e-07, 0.29610147017459754, 0.3450922758620141, 0.35880548433804277, 3.848126727713332e-07]
2 17.97 6.739999999999998 0.06739999999999999 [0.08157322437310899, 0.38227520230275797, 0.48542329275961205, 0.04990126831901309, 0.0008270122455080694] [3.848126727713332e-07, 0.29610147017459754, 0.3450922758620141, 0.35880548433804277, 3.848126727713332e-07]
3 9.079999999999998 6.739999999999998 0.06739999999999999 [0.0007063936589655386, 0.39949257213818223, 0.13297600483700434, 0.3453995494914103, 0.12142547987443754] [3.84812672771333

KeyboardInterrupt: 